# ⚡ Proactive Route & Headway Optimization
This notebook demonstrates:
1. Integrating future-hour predicted delays ($t+1$)
2. Proactively rebalancing vehicle headways before congestion peaks hit
3. Benchmarking Baseline Passenger Wait Times vs Optimized Dynamic Wait Times
4. Quantifying passenger delay reduction and fleet utilization

In [ ]:
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('..'))
from src.optimizer import DynamicRouteOptimizer

## 1. Simulate Scheduled Baseline & Predicted Delays

In [ ]:
current_schedule = pd.DataFrame({
    'route': ['22', '9', '66', '49', '79', '3', '4', '8', '60', '126'],
    'scheduled_headway_min': [10.0, 8.0, 10.0, 12.0, 10.0, 12.0, 12.0, 10.0, 12.0, 15.0],
    'scheduled_trips': [6, 8, 6, 5, 6, 5, 5, 6, 5, 4],
    'demand_index': [190, 240, 210, 130, 150, 110, 115, 140, 125, 95]
})

# Predicted delays for next peak hour (t+1h)
forecasted_delays = pd.DataFrame({
    'route': ['22', '9', '66', '49', '79', '3', '4', '8', '60', '126'],
    'forecasted_delay_min': [8.5, 9.8, 8.9, 3.5, 4.2, 3.1, 3.8, 5.2, 3.4, 2.0],
    'horizon_hours_ahead': [1] * 10
})

display(current_schedule)
display(forecasted_delays)

## 2. Execute Dynamic Headway Optimization

In [ ]:
optimizer = DynamicRouteOptimizer()
plan = optimizer.optimize_headways(current_schedule, forecasted_delays)
display(plan[['route', 'scheduled_headway_min', 'optimized_headway_min', 'baseline_avg_wait_min', 'optimized_avg_wait_min', 'wait_time_saved_min', 'pct_wait_reduction', 'action']])

## 3. Passenger Wait Time Comparison Visual

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
x = range(len(plan))
width = 0.35

ax.bar([p - width/2 for p in x], plan['baseline_avg_wait_min'], width, label='Baseline Wait Time', color='#e74c3c')
ax.bar([p + width/2 for p in x], plan['optimized_avg_wait_min'], width, label='Optimized Wait Time', color='#2ecc71')

ax.set_ylabel('Average Wait Time (minutes)', fontsize=12)
ax.set_title('Passenger Wait Time: Timetable Baseline vs Proactive Optimization', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels([f"Rt {r}" for r in plan['route']], rotation=45)
ax.legend(fontsize=11)
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

avg_savings = plan['pct_wait_reduction'].mean()
print(f"Average passenger wait time reduction across corridors: {avg_savings:.1f}%")